In [21]:
import pandas as pd
import numpy as np
import re


# Here, we compute the production, export and import of finshed and semi-finished from 2005 to 2022 using PRODCOM data and 
# JRC method (plastic and plastic embedded product), plastic categories (packaging, construction)
 
# =====================================================
# FILES
# =====================================================
estat_file   = "estat_ds-059358_en.csv.gz"
desc_file    = "PRODCOM_Description.xlsx"
table24_file = "Table24_used_share.xlsx"
output_file  = "PRODCOM_2005_2022_EU27.xlsx"

# =====================================================
# SETTINGS
# =====================================================
year_min = 2005
year_max = 2022
TARGET_YEAR = 2019

keep_inds = ["PRODQNT", "IMPQNT", "EXPQNT", "QNTUNIT", "PRODVAL", "IMPVAL", "EXPVAL"]

reporter_keep = "EU27_2020"

cols_to_split = ["PRODQNT", "IMPQNT", "EXPQNT", "PRODVAL", "IMPVAL", "EXPVAL"]
tolerance = 1e-6

# =====================================================
# COLUMN NAMES
# =====================================================
YEAR_COL   = "TIME_PERIOD"
CODE_COL   = "product"
NAME_COL   = "Full_PRODCOM_name"
SECTOR_COL = "Sector"
TYPE_COL   = "Type of product"

PROD_Q = "PRODQNT_sector"
IMP_Q  = "IMPQNT_sector"
EXP_Q  = "EXPQNT_sector"

WEIGHT_COL = "Weight"
PLSH_COL   = "Average_Plastic_share [%]"
SECSH_COL  = "Sector_share [%]"
CONV_COL   = "Conversion_factor"

# =====================================================
# HELPERS
# =====================================================
def clean_code(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    s = s.replace(".0", "")
    s = re.sub(r"\D", "", s)
    return s.zfill(8) if s else pd.NA

def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip().replace("\xa0", " ")

def norm_sector(s):
    if pd.isna(s):
        return s
    s = str(s).strip()
    low = s.lower()

    if low in ["textiles", "clothing", "clothing and textiles", "textiles and clothing"]:
        return "Clothing and textiles"
    if low in ["building & construction", "buildings and construction", "construction"]:
        return "Construction"
    if low in ["eee", "electrical and electronic equipment", "electrical & electronic equipment"]:
        return "EEE"
    return s

def normalize_share_value(x):
    if pd.isna(x):
        return 0.0
    x = float(x)
    if x > 1:
        x = x / 100.0
    return x

def is_semi(x):
    return "semi" in str(x).strip().lower()

def is_finished(x):
    t = str(x).strip().lower()
    return ("finish" in t) and ("semi" not in t)

def build_semi_category_from_name(name):
    if pd.isna(name):
        return "Other"
    s = str(name).lower()

    if any(k in s for k in ["tube", "tubes", "pipe", "pipes", "hose", "hoses"]):
        return "Tubes and pipes"
    if any(k in s for k in ["plate", "plates", "sheet", "sheets", "film", "foil", "strip"]):
        return "Plates and sheets"
    if any(k in s for k in ["fitting", "fittings", "joint", "joints", "elbow", "flange", "profile"]):
        return "Fittings and profiles"
    if any(k in s for k in ["fibre", "fiber", "yarn", "woven fabric", "woven fabrics", "non-woven", "nonwovens"]):
        return "Fibers, yarns and fabrics"
    if "net" in s or "netting" in s:
        return "Nets"
    if any(k in s for k in ["bumper", "seat belt", "wiring set", "motor vehicle"]):
        return "Automotive parts"

    return "Other"

# =====================================================
# 1) LOAD DESCRIPTION FILE = MASTER
# =====================================================
desc = pd.read_excel(desc_file)
desc.columns = desc.columns.str.strip()

desc = desc.rename(columns={"PRODCOM_codes": "product"})
desc["product"] = desc["product"].map(clean_code)

desc_cols = [
    "product",
    "Full_PRODCOM_name",
    "Sector",
    "Type of product",
    "Unit_PRODCOM_codes",
    "Weight",
    "Unit_weight",
    "Conversion_factor",
    "Unit_conversion_factor",
    "Plastic_share_[1] [%]",
    "Plastic_share_[2] [%]",
    "Average_Plastic_share [%]",
    "Sector_share [%]",
    "Comments and calculation description",
]
desc_cols = [c for c in desc_cols if c in desc.columns]

desc_master = desc[desc_cols].drop_duplicates().copy()
desc_master["Sector"] = desc_master["Sector"].map(norm_sector)

print("Rows in description file:", len(desc_master))
print("Unique product codes in description:", desc_master["product"].nunique())

# =====================================================
# 2) LOAD EUROSTAT DATA IN CHUNKS
# =====================================================
usecols = ["reporter", "product", "indicators", "TIME_PERIOD", "OBS_VALUE"]

chunks = []

for chunk in pd.read_csv(
    estat_file,
    compression="gzip",
    usecols=usecols,
    dtype={
        "reporter": "string",
        "product": "string",
        "indicators": "string",
        "TIME_PERIOD": "Int64",
        "OBS_VALUE": "string",
    },
    chunksize=500_000,
    low_memory=False,
):
    chunk["product"] = chunk["product"].map(clean_code)

    chunk = chunk[
        chunk["indicators"].isin(keep_inds)
        & chunk["TIME_PERIOD"].between(year_min, year_max)
    ].copy()

    if reporter_keep is not None:
        chunk = chunk[
            chunk["reporter"].astype(str).str.strip().str.upper() == reporter_keep.upper()
        ].copy()

    if not chunk.empty:
        chunks.append(chunk)

eur = pd.concat(chunks, ignore_index=True)

print("Filtered Eurostat rows:", len(eur))
print("Unique Eurostat product codes:", eur["product"].nunique())
print("Years:", eur["TIME_PERIOD"].min(), "to", eur["TIME_PERIOD"].max())

# =====================================================
# 3) RESHAPE EUROSTAT TO WIDE
# =====================================================
eur_wide = (
    eur.pivot_table(
        index=["reporter", "product", "TIME_PERIOD"],
        columns="indicators",
        values="OBS_VALUE",
        aggfunc="first",
    )
    .reset_index()
)

eur_wide.columns.name = None

print("Wide Eurostat rows:", len(eur_wide))
print("Wide Eurostat columns:", eur_wide.columns.tolist())

# =====================================================
# 4) MERGE WITH DESCRIPTION MASTER
# =====================================================
final = desc_master.merge(eur_wide, on="product", how="left")

print("Final rows before split:", len(final))

# =====================================================
# 5) BUILD SECTOR SHARE FRACTION
# =====================================================
sector_share_col = "Sector_share [%]"

if sector_share_col not in final.columns:
    raise KeyError(f"Column '{sector_share_col}' not found in final table.")

final[sector_share_col] = pd.to_numeric(final[sector_share_col], errors="coerce")

final["Sector_share_frac"] = np.where(
    final[sector_share_col].notna(),
    np.where(final[sector_share_col] > 1, final[sector_share_col] / 100.0, final[sector_share_col]),
    np.nan
)

final["Sector_share_frac"] = final["Sector_share_frac"].clip(lower=0, upper=1)

# =====================================================
# 6) CHECK SHARE SUM BY PRODUCT
# =====================================================
share_check = (
    final.groupby("product", dropna=False)["Sector_share_frac"]
    .sum(min_count=1)
    .reset_index(name="sector_share_sum")
)

share_check["difference_from_1"] = share_check["sector_share_sum"] - 1.0
share_check["needs_attention"] = share_check["difference_from_1"].abs() > tolerance

print("\n===== SHARE CHECK =====")
print("Products with shares not summing to 1:", share_check["needs_attention"].sum())

final = final.merge(share_check, on="product", how="left")

# =====================================================
# 7) APPLY SECTOR SPLIT ONLY WHEN NEEDED
# =====================================================
final["n_sectors_for_product"] = final.groupby("product", dropna=False)["Sector"].transform("nunique")
final["apply_sector_split"] = final["n_sectors_for_product"] > 1

existing_split = [c for c in cols_to_split if c in final.columns]

for c in existing_split:
    final[c] = pd.to_numeric(final[c], errors="coerce").astype(float)
    final[f"{c}_sector"] = final[c].astype(float)

    split_mask = final["apply_sector_split"] & final["Sector_share_frac"].notna()

    final.loc[split_mask, f"{c}_sector"] = (
        final.loc[split_mask, c].astype(float) *
        final.loc[split_mask, "Sector_share_frac"].astype(float)
    )

print("\nApplied sector split to:", existing_split)

# =====================================================
# 8) CHECK PRODUCTS WITH NO EUROSTAT MATCH
# =====================================================
coverage = desc_master.merge(
    eur_wide[["product"]].drop_duplicates(),
    on="product",
    how="left",
    indicator=True,
)

missing_products = coverage.loc[
    coverage["_merge"] == "left_only",
    ["product"]
].drop_duplicates()

if "Full_PRODCOM_name" in desc_master.columns:
    missing_products = missing_products.merge(
        desc_master[["product", "Full_PRODCOM_name"]].drop_duplicates(),
        on="product",
        how="left"
    )

print("Products with no Eurostat match (2005-2022):", len(missing_products))

# =====================================================
# 9) CONSERVATION CHECK
# =====================================================
check_rows = []
for c in existing_split:
    original_total = final[c].sum(skipna=True)
    split_total = final[f"{c}_sector"].sum(skipna=True)

    check_rows.append({
        "variable": c,
        "original_total": original_total,
        "split_total": split_total,
        "difference": split_total - original_total
    })

conservation_check = pd.DataFrame(check_rows)

print("\n===== CONSERVATION CHECK =====")
print(conservation_check)

# =====================================================
# 10) EXAMPLE CHECK
# =====================================================
example_code = "22212153"
example = final.loc[final["product"] == example_code, [
    "product", "Sector", "Sector_share [%]", "Sector_share_frac",
    "n_sectors_for_product", "apply_sector_split"
] + existing_split + [f"{c}_sector" for c in existing_split]]

if not example.empty:
    print(f"\n===== EXAMPLE {example_code} =====")
    print(example)
else:
    print(f"\nExample code {example_code} not found.")

# =====================================================
# 11) LOAD TABLE 24 FROM EXCEL
# =====================================================
table24_df = pd.read_excel(table24_file)
table24_df.columns = [str(c).strip() for c in table24_df.columns]
table24_df = table24_df.rename(columns={table24_df.columns[0]: "Category"})

table24_df.columns = [
    "Category" if c == "Category" else norm_sector(c)
    for c in table24_df.columns
]

table24_used_share = {}

for _, row in table24_df.iterrows():
    category = str(row["Category"]).strip()
    table24_used_share[category] = {}

    for col in table24_df.columns:
        if col == "Category":
            continue

        val = row[col]
        if pd.notna(val) and float(val) > 0:
            table24_used_share[category][col] = normalize_share_value(val)

def get_used_share(semi_category, sector):
    semi_category = str(semi_category).strip()
    sector = norm_sector(sector)

    if semi_category not in table24_used_share:
        return 0.0

    return float(table24_used_share[semi_category].get(sector, 0.0))

# =====================================================
# 12) BUILD MFA DATA FOR TARGET_YEAR
# =====================================================
df = final.copy()
df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce")
df = df[df[YEAR_COL] == TARGET_YEAR].copy()

if df.empty:
    raise ValueError(f"No rows found for {TARGET_YEAR}")

df[SECTOR_COL] = df[SECTOR_COL].map(norm_sector)
df[CODE_COL] = df[CODE_COL].astype(str).str.strip()

for c in [PROD_Q, IMP_Q, EXP_Q, WEIGHT_COL, PLSH_COL, SECSH_COL, CONV_COL]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    else:
        raise KeyError(f"Required column '{c}' not found in data.")

df[PLSH_COL] = df[PLSH_COL].apply(normalize_share_value)
df[SECSH_COL] = df[SECSH_COL].apply(normalize_share_value)

# optional reclassification
carpet_codes = ["13931200", "13931300", "13931990"]
mask_carpets = df[CODE_COL].isin(carpet_codes)
df.loc[mask_carpets, SECTOR_COL] = "Construction"

# =====================================================
# 13) PLASTIC MASS FLOWS
# =====================================================
df["PROD_KG_PRODUCT"] = df[PROD_Q] * df[WEIGHT_COL]
df["IMP_KG_PRODUCT"]  = df[IMP_Q]  * df[WEIGHT_COL]
df["EXP_KG_PRODUCT"]  = df[EXP_Q]  * df[WEIGHT_COL]

df["plastic_frac"] = df[PLSH_COL]
df["sector_frac"]  = df[SECSH_COL]

# Keep your original conversion logic
df["conv_Mt_per_kg"] = df[CONV_COL] * 0.001

# You can include sector_frac here if needed
df["PROD_MT_PL"] = df["PROD_KG_PRODUCT"] * df["conv_Mt_per_kg"] * df["plastic_frac"]
df["IMP_MT_PL"]  = df["IMP_KG_PRODUCT"]  * df["conv_Mt_per_kg"] * df["plastic_frac"]
df["EXP_MT_PL"]  = df["EXP_KG_PRODUCT"]  * df["conv_Mt_per_kg"] * df["plastic_frac"]

# =====================================================
# 14) CLASSIFY
# =====================================================
df["semi_category"] = df[NAME_COL].apply(build_semi_category_from_name)
df["is_semi"] = df[TYPE_COL].apply(is_semi)
df["is_finished"] = df[TYPE_COL].apply(is_finished)

# =====================================================
# 15) APPLY TABLE 24 SPLIT
# =====================================================
mask_semi = df["is_semi"].fillna(False)

df["semi_used_share"] = 0.0
df.loc[mask_semi, "semi_used_share"] = df.loc[mask_semi].apply(
    lambda r: get_used_share(r["semi_category"], r[SECTOR_COL]),
    axis=1
)

df["semi_used_share"] = df["semi_used_share"].clip(0, 1)
df["semi_direct_share"] = 1.0
df.loc[mask_semi, "semi_direct_share"] = 1.0 - df.loc[mask_semi, "semi_used_share"]

for flow in ["PROD_MT_PL", "IMP_MT_PL", "EXP_MT_PL"]:
    df[f"SEMI_TO_FINISHED_{flow}"] = 0.0
    df[f"SEMI_DIRECT_{flow}"] = 0.0

    df.loc[mask_semi, f"SEMI_TO_FINISHED_{flow}"] = (
        df.loc[mask_semi, flow] * df.loc[mask_semi, "semi_used_share"]
    )
    df.loc[mask_semi, f"SEMI_DIRECT_{flow}"] = (
        df.loc[mask_semi, flow] * df.loc[mask_semi, "semi_direct_share"]
    )

# =====================================================
# 16) FINISHED ONLY FLOWS
# =====================================================
df["FINISHED_ONLY_PROD_MT"] = 0.0
df["FINISHED_ONLY_IMP_MT"]  = 0.0
df["FINISHED_ONLY_EXP_MT"]  = 0.0

df.loc[df["is_finished"], "FINISHED_ONLY_PROD_MT"] = df.loc[df["is_finished"], "PROD_MT_PL"]
df.loc[df["is_finished"], "FINISHED_ONLY_IMP_MT"]  = df.loc[df["is_finished"], "IMP_MT_PL"]
df.loc[df["is_finished"], "FINISHED_ONLY_EXP_MT"]  = df.loc[df["is_finished"], "EXP_MT_PL"]

# =====================================================
# 17) BUILD PRODCOM MFA TOTALS
# =====================================================

df["SEMI_DIRECT_CONS_MT"] = 0.0
df.loc[df["is_semi"], "SEMI_DIRECT_CONS_MT"] = (
    df.loc[df["is_semi"], "SEMI_DIRECT_PROD_MT_PL"]
    + df.loc[df["is_semi"], "SEMI_DIRECT_IMP_MT_PL"]
    - df.loc[df["is_semi"], "SEMI_DIRECT_EXP_MT_PL"]
)


df["SEMI_TO_FINISHED_CONS_MT"] = (
    df["SEMI_TO_FINISHED_PROD_MT_PL"]
    + df["SEMI_TO_FINISHED_IMP_MT_PL"]
    - df["SEMI_TO_FINISHED_EXP_MT_PL"]
)

df["FINISHED_TOTAL_PROD_MT"] = (
    df["FINISHED_ONLY_PROD_MT"] + df["SEMI_TO_FINISHED_PROD_MT_PL"]
)


df["FINISHED_TOTAL_EXP_MT"] = (
    df["FINISHED_ONLY_EXP_MT"] + df["SEMI_TO_FINISHED_EXP_MT_PL"]
)

df["FINISHED_TOTAL_IMP_MT"] = (
    df["FINISHED_ONLY_IMP_MT"] + df["SEMI_TO_FINISHED_IMP_MT_PL"]
)

df["FINISHED_EFFECTIVE_CONS_MT"] = (
    df["FINISHED_TOTAL_PROD_MT"]+ df["FINISHED_TOTAL_IMP_MT"] - df["FINISHED_TOTAL_EXP_MT"]
)

df["CONS_FINAL_MT"] = (
    df["FINISHED_EFFECTIVE_CONS_MT"] + df["SEMI_DIRECT_CONS_MT"]
)

# =====================================================
# 18) DIAGNOSTICS
# =====================================================
diag_cols = [
    "SEMI_DIRECT_CONS_MT",
    "SEMI_TO_FINISHED_CONS_MT",
    "FINISHED_EFFECTIVE_CONS_MT",
    "CONS_FINAL_MT"
]

for c in diag_cols:
    df[f"{c}_NEGATIVE"] = df[c] < 0

totals = df[[
    "PROD_MT_PL",
    "IMP_MT_PL",
    "EXP_MT_PL",
    "SEMI_DIRECT_CONS_MT",
    "SEMI_TO_FINISHED_CONS_MT",
    "FINISHED_TOTAL_PROD_MT",
    "FINISHED_TOTAL_IMP_MT",
    "FINISHED_TOTAL_EXP_MT",
    "CONS_FINAL_MT"
]].sum().to_frame("Mt")

# =====================================================
# 19B) PRODUCTION / IMPORT / EXPORT BY PRODUCT TYPE AND SECTOR
# =====================================================

# 1) By sector and type of product
Type_trade_by_sector = (
    df.groupby([SECTOR_COL, TYPE_COL], dropna=False)[
        ["PROD_MT_PL", "IMP_MT_PL", "EXP_MT_PL"]
    ]
    .sum()
    .reset_index()
    .rename(columns={
        "PROD_MT_PL": "Production_MT",
        "IMP_MT_PL": "Import_MT",
        "EXP_MT_PL": "Export_MT"
    })
)

print("\n===== PRODUCTION / IMPORT / EXPORT BY SECTOR AND TYPE =====")
print(Type_trade_by_sector)

# =====================================================
# 19) FINISHED CATEGORY = FINISHED ONLY + SEMI TO FINISHED
# =====================================================

# Finished-only flows by sector
Finished_only_by_sector = (
    df.groupby(SECTOR_COL, dropna=False)[
        ["FINISHED_ONLY_PROD_MT", "FINISHED_ONLY_IMP_MT", "FINISHED_ONLY_EXP_MT"]
    ]
    .sum()
    .reset_index()
    .rename(columns={
        "FINISHED_ONLY_PROD_MT": "Finished_only_production_MT",
        "FINISHED_ONLY_IMP_MT": "Finished_only_import_MT",
        "FINISHED_ONLY_EXP_MT": "Finished_only_export_MT"
    })
)

# Semi-finished flows used in finished production by sector
Semi_to_finished_by_sector = (
    df.groupby(SECTOR_COL, dropna=False)[
        [
            "SEMI_TO_FINISHED_PROD_MT_PL",
            "SEMI_TO_FINISHED_IMP_MT_PL",
            "SEMI_TO_FINISHED_EXP_MT_PL"
        ]
    ]
    .sum()
    .reset_index()
    .rename(columns={
        "SEMI_TO_FINISHED_PROD_MT_PL": "Semi_to_finished_production_MT",
        "SEMI_TO_FINISHED_IMP_MT_PL": "Semi_to_finished_import_MT",
        "SEMI_TO_FINISHED_EXP_MT_PL": "Semi_to_finished_export_MT"
    })
)

# Merge both
Finished_category_by_sector = (
    Finished_only_by_sector
    .merge(Semi_to_finished_by_sector, on=SECTOR_COL, how="outer")
    .fillna(0)
)

# Final finished category = finished only + semi to finished
Finished_category_by_sector["Finished_category_production_MT"] = (
    Finished_category_by_sector["Finished_only_production_MT"]
    + Finished_category_by_sector["Semi_to_finished_production_MT"]
)

Finished_category_by_sector["Finished_category_import_MT"] = (
    Finished_category_by_sector["Finished_only_import_MT"]
    + Finished_category_by_sector["Semi_to_finished_import_MT"]
)

Finished_category_by_sector["Finished_category_export_MT"] = (
    Finished_category_by_sector["Finished_only_export_MT"]
    + Finished_category_by_sector["Semi_to_finished_export_MT"]
)

print("\n===== FINISHED CATEGORY BY SECTOR =====")
print(Finished_category_by_sector)

# =====================================================
# 19D) TOTAL CHECKS
# =====================================================

Finished_category_totals = pd.DataFrame({
    "Flow": ["Production", "Import", "Export"],
    "Finished_only_MT": [
        Finished_category_by_sector["Finished_only_production_MT"].sum(),
        Finished_category_by_sector["Finished_only_import_MT"].sum(),
        Finished_category_by_sector["Finished_only_export_MT"].sum()
    ],
    "Semi_to_finished_MT": [
        Finished_category_by_sector["Semi_to_finished_production_MT"].sum(),
        Finished_category_by_sector["Semi_to_finished_import_MT"].sum(),
        Finished_category_by_sector["Semi_to_finished_export_MT"].sum()
    ],
    "Finished_category_total_MT": [
        Finished_category_by_sector["Finished_category_production_MT"].sum(),
        Finished_category_by_sector["Finished_category_import_MT"].sum(),
        Finished_category_by_sector["Finished_category_export_MT"].sum()
    ]
})

print("\n===== FINISHED CATEGORY TOTALS =====")
print(Finished_category_totals)

# =====================================================
#19 IMPORT / EXPORT OF SEMI AND FINISHED BY SECTOR
# =====================================================
Trade_by_sector_type = (
    df.groupby([SECTOR_COL, TYPE_COL], dropna=False)[["IMP_MT_PL", "EXP_MT_PL"]].sum().reset_index() )
print("\n===== IMPORT / EXPORT BY SECTOR AND TYPE OF PRODUCT =====")
print(Trade_by_sector_type)

# -----------------------------------------------------
# Semi-finished only
# -----------------------------------------------------
Semi_trade_by_sector = (
    df[df["is_semi"]]
    .groupby(SECTOR_COL, dropna=False)[["IMP_MT_PL", "EXP_MT_PL"]]
    .sum()
    .reset_index()
    .rename(columns={
        "IMP_MT_PL": "Import_semi_MT",
        "EXP_MT_PL": "Export_semi_MT"
    })
)
print("\n===== SEMI-FINISHED IMPORT / EXPORT BY SECTOR =====")
print(Semi_trade_by_sector)
# -----------------------------------------------------
# Finished only
# -----------------------------------------------------
Finished_trade_by_sector = (
    df[df["is_finished"]]
    .groupby(SECTOR_COL, dropna=False)[["IMP_MT_PL", "EXP_MT_PL"]]
    .sum()
    .reset_index()
    .rename(columns={
        "IMP_MT_PL": "Import_finished_MT",
        "EXP_MT_PL": "Export_finished_MT"
    })
)
# Merge semi + finished in one table
# -----------------------------------------------------
Trade_summary = (
    pd.merge(Semi_trade_by_sector, Finished_trade_by_sector, on=SECTOR_COL, how="outer")
      .fillna(0)
)

Trade_summary["Import_total_MT"] = (
    Trade_summary["Import_semi_MT"] + Trade_summary["Import_finished_MT"]
)
Trade_summary["Export_total_MT"] = (
    Trade_summary["Export_semi_MT"] + Trade_summary["Export_finished_MT"]
)

print("\n===== FINAL TRADE SUMMARY BY SECTOR =====")
print(Trade_summary)

Trade_totals = pd.DataFrame({
    "Category": ["Semi-finished", "Finished"],
    "Imports_MT": [
        df.loc[df["is_semi"], "IMP_MT_PL"].sum(),
        df.loc[df["is_finished"], "IMP_MT_PL"].sum()
    ],
    "Exports_MT": [
        df.loc[df["is_semi"], "EXP_MT_PL"].sum(),
        df.loc[df["is_finished"], "EXP_MT_PL"].sum()
    ]
})

print("\n===== TOTAL IMPORTS / EXPORTS =====")
print(Trade_totals)

# =====================================================
# 20) MASS BALANCE CHECK
# =====================================================
balance = pd.DataFrame({
    "Metric": [
        "Total production",
        "Total imports",
        "Total exports",
        "Apparent consumption = Prod + Imp - Exp",
        "Computed final consumption"
    ],
    "Mt": [
        df["PROD_MT_PL"].sum(),
        df["IMP_MT_PL"].sum(),
        df["EXP_MT_PL"].sum(),
        df["PROD_MT_PL"].sum() + df["IMP_MT_PL"].sum() - df["EXP_MT_PL"].sum(),
        df["CONS_FINAL_MT"].sum()
    ]
})

balance["Difference_vs_computed"] = np.nan
balance.loc[3, "Difference_vs_computed"] = balance.loc[3, "Mt"] - balance.loc[4, "Mt"]

# =====================================================
# 20) SAVE
# =====================================================
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    final.to_excel(writer, sheet_name="data", index=False)
    missing_products.to_excel(writer, sheet_name="missing_products", index=False)
    share_check.to_excel(writer, sheet_name="share_check", index=False)
    conservation_check.to_excel(writer, sheet_name="conservation_check", index=False)

    df.to_excel(writer, sheet_name=f"mfa_{TARGET_YEAR}", index=False)
    totals.to_excel(writer, sheet_name="mfa_totals")
    balance.to_excel(writer, sheet_name="mass_balance_check", index=False)


    Type_trade_by_sector.to_excel(writer, sheet_name="Type_by_sector", index=False)
    Finished_only_by_sector.to_excel(writer, sheet_name="Finished_only_sector", index=False)
    Semi_to_finished_by_sector.to_excel(writer, sheet_name="Semi_to_finished_sector", index=False)
    Finished_category_by_sector.to_excel(writer, sheet_name="Finished_category_sector", index=False)
    Finished_category_totals.to_excel(writer, sheet_name="Finished_category_totals", index=False)
    
    Trade_by_sector_type.to_excel(writer, sheet_name="Trade_by_sector_type", index=False)
    Semi_trade_by_sector.to_excel(writer, sheet_name="Semi_trade_by_sector", index=False)
    Finished_trade_by_sector.to_excel(writer, sheet_name="Finished_trade_by_sector", index=False)
    Trade_summary.to_excel(writer, sheet_name="Trade_summary_sector", index=False)
    Trade_totals.to_excel(writer, sheet_name="Trade_totals", index=False)

print("Saved:", output_file)
print("\nMFA totals:")
print(totals)
print("\nMass balance check:")
print(balance)

Rows in description file: 577
Unique product codes in description: 458
Filtered Eurostat rows: 393127
Unique Eurostat product codes: 4785
Years: 2005 to 2022
Wide Eurostat rows: 68916
Wide Eurostat columns: ['reporter', 'product', 'TIME_PERIOD', 'EXPQNT', 'EXPVAL', 'IMPQNT', 'IMPVAL', 'PRODQNT', 'PRODVAL', 'QNTUNIT']
Final rows before split: 10011

===== SHARE CHECK =====
Products with shares not summing to 1: 458

Applied sector split to: ['PRODQNT', 'IMPQNT', 'EXPQNT', 'PRODVAL', 'IMPVAL', 'EXPVAL']
Products with no Eurostat match (2005-2022): 1

===== CONSERVATION CHECK =====
  variable  original_total   split_total    difference
0  PRODQNT    1.643962e+13  1.542260e+13 -1.017021e+12
1   IMPQNT    8.920548e+11  7.891256e+11 -1.029292e+11
2   EXPQNT    2.697966e+12  2.550318e+12 -1.476477e+11
3  PRODVAL    1.468471e+13  1.204056e+13 -2.644151e+12
4   IMPVAL    4.808006e+12  4.406660e+12 -4.013460e+11
5   EXPVAL    5.309005e+12  4.698809e+12 -6.101957e+11

===== EXAMPLE 22212153 =====

In [ ]:
import pandas as pd
import numpy as np


# Here, we compute the TC for export and import of finshed and semi-finished from 2005 to 2022 
# ==========================================================
# FILES
# ==========================================================
trade_file = "PRODCOM_2005_2022_EU27.xlsx"
plastic_demand_file = "Plastic_demand.xlsx"
sector_alloc_file   = "Sector_allocation.xlsx"
tc_domestic_file    = "TC_semi_Finished.xlsx"

out_file = "TC_Trade_Finished_Semi_By_Sector_EU_27_2019.xlsx"

# ==========================================================
# SETTINGS
# ==========================================================
trade_sheet = "mfa_2019"
# ==========================================================
# HELPERS
# ==========================================================
def norm_sector(s):
    s = str(s).strip()
    low = s.lower()

    if low in ["clothing_textiles", "clothing & textiles", "textiles", "clothing",
               "textiles and clothing", "clothing and textiles", "textiles&clothing"]:
        return "Clothing and textiles"
    if low in ["building & construction", "buildings and construction", "construction", "building"]:
        return "Construction"
    if low in ["agriculture", "agriculture, farming, gardening"]:
        return "Agriculture"
    if low in ["eee", "electrical and electronic equipment", "electrical/electronics",
               "electrical & electronic equipment"]:
        return "EEE"
    if low in ["healthcare", "health care"]:
        return "Healthcare"
    if low in ["fishing"]:
        return "Fishing"
    if low in ["other", "others"]:
        return "Other"
    if low in ["packaging"]:
        return "Packaging"
    if low in ["transport", "automotive"]:
        return "Transport"

    return s


def product_type_group(x):
    x = str(x).lower().strip()

    if "semi" in x:
        return "semi"
    if "finish" in x and "semi" not in x:
        return "finished"

    return "other"


def to_float(x):
    if pd.isna(x):
        return 0.0

    if isinstance(x, str):
        x = x.strip().replace("%", "")
        if x == "":
            return 0.0

    return float(x)


def clean_text(x):
    if pd.isna(x):
        return ""
    return " ".join(str(x).replace("\n", " ").replace("\r", " ").split()).strip().lower()


sector_to_code = {
    "Packaging": "P",
    "Construction": "C",
    "Transport": "T",
    "EEE": "E",
    "Agriculture": "A",
    "Clothing and textiles": "C&T",
    "Healthcare": "H",
    "Fishing": "F",
    "Other": "O"
}

sector_order = ["P", "C", "T", "E", "A", "C&T", "H", "F", "O"]

code_to_sector = {v: k for k, v in sector_to_code.items()}

# ==========================================================
# 1) LOAD TRADE DATA
# ==========================================================
trade = pd.read_excel(trade_file, sheet_name=trade_sheet)
trade.columns = trade.columns.str.strip()

required_trade_cols = ["Sector", "Type of product", "IMP_MT_PL", "EXP_MT_PL"]
missing_trade = [c for c in required_trade_cols if c not in trade.columns]

if missing_trade:
    raise KeyError(f"Missing columns in trade file: {missing_trade}")

trade = trade[required_trade_cols].copy()

trade["Sector"] = trade["Sector"].apply(norm_sector)
trade["Sector_code"] = trade["Sector"].map(sector_to_code)

trade["Type_group"] = trade["Type of product"].apply(product_type_group)

trade["IMP_MT_PL"] = pd.to_numeric(trade["IMP_MT_PL"], errors="coerce").fillna(0.0)
trade["EXP_MT_PL"] = pd.to_numeric(trade["EXP_MT_PL"], errors="coerce").fillna(0.0)

# Keep only semi and finished
trade = trade[trade["Type_group"].isin(["semi", "finished"])].copy()

# Aggregate trade by sector + type
trade_agg = (
    trade
    .groupby(["Sector", "Sector_code", "Type_group"], dropna=False)[["IMP_MT_PL", "EXP_MT_PL"]]
    .sum()
    .reset_index()
)

# ==========================================================
# 2) BUILD DOMESTIC INPUT DATA FROM:
# Plastic demand × sector allocation share × TC
# ==========================================================

# ----------------------------------------------------------
# 2.1 Load plastic demand
# ----------------------------------------------------------
plastic_demand_df = pd.read_excel(plastic_demand_file)
plastic_demand_df.columns = plastic_demand_df.columns.str.strip()

# Try to detect the demand column automatically
possible_demand_cols = [
    "Plastic_demand_MT",
    "Plastic demand",
    "Plastic demand MT",

]

demand_col = None
for c in possible_demand_cols:
    if c in plastic_demand_df.columns:
        demand_col = c
        break

if demand_col is None:
    numeric_cols = plastic_demand_df.select_dtypes(include="number").columns.tolist()
    if len(numeric_cols) == 0:
        raise KeyError("No plastic demand column found in Plastic_demand.xlsx.")
    demand_col = numeric_cols[0]

Plastic_demand_MT = float(plastic_demand_df[demand_col].dropna().iloc[0])

print("\nPlastic demand used:", Plastic_demand_MT, "Mt")

# ----------------------------------------------------------
# 2.2 Load allocation shares by sector
alloc = pd.read_excel(sector_alloc_file, header=None)

# sector names are in row 0, from column B onward
sector_names = alloc.iloc[0, 1:].dropna().tolist()

# allocation values are in row 1, from column B onward
allocation_values = alloc.iloc[1, 1:1 + len(sector_names)].tolist()

sector_alloc = pd.DataFrame({
    "Sector": sector_names,
    "Allocation_share": allocation_values
})

sector_alloc["Sector"] = sector_alloc["Sector"].apply(norm_sector)
sector_alloc["Sector_code"] = sector_alloc["Sector"].map(sector_to_code)

sector_alloc["Allocation_share"] = pd.to_numeric(
    sector_alloc["Allocation_share"], errors="coerce"
).fillna(0.0)

# If shares are written as percentages, convert to fractions
sector_alloc["Allocation_share"] = np.where(
    sector_alloc["Allocation_share"] > 1,
    sector_alloc["Allocation_share"] / 100,
    sector_alloc["Allocation_share"]
)

sector_alloc = sector_alloc[sector_alloc["Sector_code"].isin(sector_order)].copy()

print("\n===== SECTOR ALLOCATION SHARES =====")
print(sector_alloc)

# ----------------------------------------------------------
# 2.3 Load TC file for domestic split: plastic -> semi / finished
# ----------------------------------------------------------
tc_dom = pd.read_excel(tc_domestic_file)
tc_dom.columns = tc_dom.columns.str.strip()

required_tc_cols = ["FROM", "TO"]
missing_tc_cols = [c for c in required_tc_cols if c not in tc_dom.columns]

if missing_tc_cols:
    raise KeyError(f"Missing columns in TC file: {missing_tc_cols}")

tc_dom["_FROM_CLEAN"] = tc_dom["FROM"].apply(clean_text)
tc_dom["_TO_CLEAN"] = tc_dom["TO"].apply(clean_text)

semi_mask = (
    (tc_dom["_FROM_CLEAN"] == "plastic products manufacturing") &
    (tc_dom["_TO_CLEAN"] == "semi-finished products manufacturing")
)

finished_mask = (
    (tc_dom["_FROM_CLEAN"] == "plastic products manufacturing") &
    (tc_dom["_TO_CLEAN"] == "finished products manufacturing")
)

if not semi_mask.any():
    raise ValueError("Row not found: Plastic products manufacturing -> Semi-finished products manufacturing")

if not finished_mask.any():
    raise ValueError("Row not found: Plastic products manufacturing -> Finished products manufacturing")

semi_row = tc_dom.loc[semi_mask].iloc[0]
finished_row = tc_dom.loc[finished_mask].iloc[0]

# Build a flexible sector-column map from TC file columns
tc_sector_col_map = {}

for col in tc_dom.columns:
    if col in ["FROM", "TO", "_FROM_CLEAN", "_TO_CLEAN"]:
        continue

    col_clean = norm_sector(col)

    if col_clean in sector_to_code:
        tc_sector_col_map[sector_to_code[col_clean]] = col
    elif col in sector_order:
        tc_sector_col_map[col] = col

print("\n===== TC SECTOR COLUMN MAP =====")
print(tc_sector_col_map)

missing_tc_sector_cols = [s for s in sector_order if s not in tc_sector_col_map]
if missing_tc_sector_cols:
    raise KeyError(
        f"These sector columns are missing in TC file: {missing_tc_sector_cols}. "
        f"Available TC columns are: {tc_dom.columns.tolist()}"
    )

# ----------------------------------------------------------
# 2.4 Build domestic input table
# ----------------------------------------------------------
rows = []

for _, r in sector_alloc.iterrows():

    sector_code = r["Sector_code"]
    sector_name = code_to_sector.get(sector_code, r.get("Sector", sector_code))
    alloc_share = float(r["Allocation_share"])

    tc_col = tc_sector_col_map[sector_code]

    tc_semi = to_float(semi_row[tc_col])
    tc_finished = to_float(finished_row[tc_col])

    # Convert TC if written as %
    if tc_semi > 1:
        tc_semi = tc_semi / 100

    if tc_finished > 1:
        tc_finished = tc_finished / 100

    semi_dom = Plastic_demand_MT * alloc_share * tc_semi
    finished_dom = Plastic_demand_MT * alloc_share * tc_finished

    rows.append({
        "Sector": sector_name,
        "Sector_code": sector_code,
        "Plastic_demand_MT": Plastic_demand_MT,
        "Allocation_share": alloc_share,
        "TC_plastic_to_semi": tc_semi,
        "TC_plastic_to_finished": tc_finished,
        "Semi_dom_MT": semi_dom,
        "Finished_domestic_MT": finished_dom
    })

dom_use_detail = pd.DataFrame(rows)

dom_use = (
    dom_use_detail
    .groupby("Sector_code", dropna=False)[["Semi_dom_MT", "Finished_domestic_MT"]]
    .sum()
    .reset_index()
)

print("\n===== DOMESTIC INPUTS BUILT FROM DEMAND × ALLOCATION × TC =====")
print(dom_use_detail)

# ==========================================================
# 3) MERGE TRADE + DOMESTIC INPUT
# ==========================================================
tc = trade_agg.merge(dom_use, on="Sector_code", how="left")

tc["Domestic_input_MT"] = np.where(
    tc["Type_group"] == "semi",
    tc["Semi_dom_MT"],
    tc["Finished_domestic_MT"]
)
# ==========================================================
# 4) COMPUTE JRC TRADE TCs
# ==========================================================

tc["TC_Import"] = np.where(
    tc["Domestic_input_MT"] > 0,
    tc["IMP_MT_PL"] / tc["Domestic_input_MT"],
    np.nan
)

tc["TC_Export"] = np.where(
    tc["Domestic_input_MT"] > 0,
    tc["EXP_MT_PL"] / tc["Domestic_input_MT"],
    np.nan
)

tc["TC_Import"] = tc["TC_Import"].round(2)
tc["TC_Export"] = tc["TC_Export"].round(2)

# Add readable type
tc["Type of product"] = np.where(
    tc["Type_group"] == "semi",
    "Semi-finished",
    "Finished"
)

# ==========================================================
# 5) LONG FORMAT TABLE
# ==========================================================
tc_final = tc[[
    "Sector",
    "Sector_code",
    "Type of product",
    "Type_group",
    "IMP_MT_PL",
    "EXP_MT_PL",
    "Domestic_input_MT",
    "TC_Import",
    "TC_Export"
]].sort_values(["Sector_code", "Type_group"]).reset_index(drop=True)

print("\n===== LONG TABLE: JRC TRADE TCs BY SECTOR =====")
print(tc_final)

# ==========================================================
# 6) BUILD JRC-STYLE MATRIX
# ==========================================================
jrc_rows = [
    (
        "Import",
        "Semi-finished products manufacturing",
        "TC_Import",
        "semi"
    ),
    (
        "Import",
        "Finished products manufacturing",
        "TC_Import",
        "finished"
    ),
    (
        "Semi-finished products manufacturing",
        "Export",
        "TC_Export",
        "semi"
    ),
    (
        "Finished products manufacturing",
        "Export",
        "TC_Export",
        "finished"
    ),
    (
        "Plastic products manufacturing",
        "Semi-finished products manufacturing (domestic input Semi_dom_MT)",
        "Domestic_input_MT",
        "semi"
    ),
    (
        "Plastic products manufacturing",
        "Finished products manufacturing (domestic input Finished_domestic_MT)",
        "Domestic_input_MT",
        "finished"
    )
]

matrix_rows = []

for from_node, to_node, value_col, group in jrc_rows:

    row = {
        "FROM": from_node,
        "TO": to_node
    }

    sub = tc_final[tc_final["Type_group"] == group].copy()

    for sec in sector_order:
        value = sub.loc[sub["Sector_code"] == sec, value_col]

        if len(value) > 0:
            row[sec] = value.iloc[0]
        else:
            row[sec] = np.nan

    matrix_rows.append(row)

jrc_matrix = pd.DataFrame(matrix_rows)

print("\n===== JRC-STYLE MATRIX =====")
print(jrc_matrix)

# ==========================================================
# 7) OPTIONAL: SEPARATE TABLES
# ==========================================================
semi_table = tc_final[tc_final["Type_group"] == "semi"].copy()
finished_table = tc_final[tc_final["Type_group"] == "finished"].copy()

# ==========================================================
# 8) SAVE OUTPUT
# ==========================================================
with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    tc_final.to_excel(writer, sheet_name="TC_trade_long", index=False)
    semi_table.to_excel(writer, sheet_name="TC_semi", index=False)
    finished_table.to_excel(writer, sheet_name="TC_finished", index=False)
    jrc_matrix.to_excel(writer, sheet_name="JRC_style_matrix", index=False)
    dom_use_detail.to_excel(writer, sheet_name="Domestic_inputs_calc", index=False)

print(f"\nSaved: {out_file}")


Plastic demand used: 53.3 Mt

===== SECTOR ALLOCATION SHARES =====
                  Sector  Allocation_share Sector_code
0              Packaging            0.3960           P
1           Construction            0.2040           C
2              Transport            0.0960           T
3                    EEE            0.0620           E
4            Agriculture            0.0340           A
5  Clothing and textiles            0.0339         C&T
6             Healthcare            0.0016           H
7                Fishing            0.0029           F
8                  Other            0.1696           O

===== TC SECTOR COLUMN MAP =====
{'P': 'Packaging', 'C': 'Construction', 'T': 'Transport', 'E': 'EEE', 'A': 'Agriculture', 'C&T': 'Clothing and textiles', 'H': 'Healthcare', 'F': 'Fishing', 'O': 'Other'}

===== DOMESTIC INPUTS BUILT FROM DEMAND × ALLOCATION × TC =====
                  Sector Sector_code  Plastic_demand_MT  Allocation_share  \
0              Packaging           

In [28]:
import pandas as pd
import numpy as np
import re


# Here, we compute the production, export and import of finshed and semi-finished from 2005 to 2022 using PRODCOM data and 
# JRC method (plastic and plastic embedded product), plastic categories (packaging, construction)
 
# =====================================================
# FILES
# =====================================================
estat_file   = "estat_ds-059358_en.csv.gz"
desc_file    = "PRODCOM_Description.xlsx"
table24_file = "Table24_used_share.xlsx"
output_file  = "PRODCOM_2005_2022_EU27_2022.xlsx"

# =====================================================
# SETTINGS
# =====================================================
year_min = 2005
year_max = 2022
TARGET_YEAR = 2022

keep_inds = ["PRODQNT", "IMPQNT", "EXPQNT", "QNTUNIT", "PRODVAL", "IMPVAL", "EXPVAL"]

reporter_keep = "EU27_2020"

cols_to_split = ["PRODQNT", "IMPQNT", "EXPQNT", "PRODVAL", "IMPVAL", "EXPVAL"]
tolerance = 1e-6

# =====================================================
# COLUMN NAMES
# =====================================================
YEAR_COL   = "TIME_PERIOD"
CODE_COL   = "product"
NAME_COL   = "Full_PRODCOM_name"
SECTOR_COL = "Sector"
TYPE_COL   = "Type of product"

PROD_Q = "PRODQNT_sector"
IMP_Q  = "IMPQNT_sector"
EXP_Q  = "EXPQNT_sector"

WEIGHT_COL = "Weight"
PLSH_COL   = "Average_Plastic_share [%]"
SECSH_COL  = "Sector_share [%]"
CONV_COL   = "Conversion_factor"

# =====================================================
# HELPERS
# =====================================================
def clean_code(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    s = s.replace(".0", "")
    s = re.sub(r"\D", "", s)
    return s.zfill(8) if s else pd.NA

def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip().replace("\xa0", " ")

def norm_sector(s):
    if pd.isna(s):
        return s
    s = str(s).strip()
    low = s.lower()

    if low in ["textiles", "clothing", "clothing and textiles", "textiles and clothing"]:
        return "Clothing and textiles"
    if low in ["building & construction", "buildings and construction", "construction"]:
        return "Construction"
    if low in ["eee", "electrical and electronic equipment", "electrical & electronic equipment"]:
        return "EEE"
    return s

def normalize_share_value(x):
    if pd.isna(x):
        return 0.0
    x = float(x)
    if x > 1:
        x = x / 100.0
    return x

def is_semi(x):
    return "semi" in str(x).strip().lower()

def is_finished(x):
    t = str(x).strip().lower()
    return ("finish" in t) and ("semi" not in t)

def build_semi_category_from_name(name):
    if pd.isna(name):
        return "Other"
    s = str(name).lower()

    if any(k in s for k in ["tube", "tubes", "pipe", "pipes", "hose", "hoses"]):
        return "Tubes and pipes"
    if any(k in s for k in ["plate", "plates", "sheet", "sheets", "film", "foil", "strip"]):
        return "Plates and sheets"
    if any(k in s for k in ["fitting", "fittings", "joint", "joints", "elbow", "flange", "profile"]):
        return "Fittings and profiles"
    if any(k in s for k in ["fibre", "fiber", "yarn", "woven fabric", "woven fabrics", "non-woven", "nonwovens"]):
        return "Fibers, yarns and fabrics"
    if "net" in s or "netting" in s:
        return "Nets"
    if any(k in s for k in ["bumper", "seat belt", "wiring set", "motor vehicle"]):
        return "Automotive parts"

    return "Other"

# =====================================================
# 1) LOAD DESCRIPTION FILE = MASTER
# =====================================================
desc = pd.read_excel(desc_file)
desc.columns = desc.columns.str.strip()

desc = desc.rename(columns={"PRODCOM_codes": "product"})
desc["product"] = desc["product"].map(clean_code)

desc_cols = [
    "product",
    "Full_PRODCOM_name",
    "Sector",
    "Type of product",
    "Unit_PRODCOM_codes",
    "Weight",
    "Unit_weight",
    "Conversion_factor",
    "Unit_conversion_factor",
    "Plastic_share_[1] [%]",
    "Plastic_share_[2] [%]",
    "Average_Plastic_share [%]",
    "Sector_share [%]",
    "Comments and calculation description",
]
desc_cols = [c for c in desc_cols if c in desc.columns]

desc_master = desc[desc_cols].drop_duplicates().copy()
desc_master["Sector"] = desc_master["Sector"].map(norm_sector)

print("Rows in description file:", len(desc_master))
print("Unique product codes in description:", desc_master["product"].nunique())

# =====================================================
# 2) LOAD EUROSTAT DATA IN CHUNKS
# =====================================================
usecols = ["reporter", "product", "indicators", "TIME_PERIOD", "OBS_VALUE"]

chunks = []

for chunk in pd.read_csv(
    estat_file,
    compression="gzip",
    usecols=usecols,
    dtype={
        "reporter": "string",
        "product": "string",
        "indicators": "string",
        "TIME_PERIOD": "Int64",
        "OBS_VALUE": "string",
    },
    chunksize=500_000,
    low_memory=False,
):
    chunk["product"] = chunk["product"].map(clean_code)

    chunk = chunk[
        chunk["indicators"].isin(keep_inds)
        & chunk["TIME_PERIOD"].between(year_min, year_max)
    ].copy()

    if reporter_keep is not None:
        chunk = chunk[
            chunk["reporter"].astype(str).str.strip().str.upper() == reporter_keep.upper()
        ].copy()

    if not chunk.empty:
        chunks.append(chunk)

eur = pd.concat(chunks, ignore_index=True)

print("Filtered Eurostat rows:", len(eur))
print("Unique Eurostat product codes:", eur["product"].nunique())
print("Years:", eur["TIME_PERIOD"].min(), "to", eur["TIME_PERIOD"].max())

# =====================================================
# 3) RESHAPE EUROSTAT TO WIDE
# =====================================================
eur_wide = (
    eur.pivot_table(
        index=["reporter", "product", "TIME_PERIOD"],
        columns="indicators",
        values="OBS_VALUE",
        aggfunc="first",
    )
    .reset_index()
)

eur_wide.columns.name = None

print("Wide Eurostat rows:", len(eur_wide))
print("Wide Eurostat columns:", eur_wide.columns.tolist())

# =====================================================
# 4) MERGE WITH DESCRIPTION MASTER
# =====================================================
final = desc_master.merge(eur_wide, on="product", how="left")

print("Final rows before split:", len(final))

# =====================================================
# 5) BUILD SECTOR SHARE FRACTION
# =====================================================
sector_share_col = "Sector_share [%]"

if sector_share_col not in final.columns:
    raise KeyError(f"Column '{sector_share_col}' not found in final table.")

final[sector_share_col] = pd.to_numeric(final[sector_share_col], errors="coerce")

final["Sector_share_frac"] = np.where(
    final[sector_share_col].notna(),
    np.where(final[sector_share_col] > 1, final[sector_share_col] / 100.0, final[sector_share_col]),
    np.nan
)

final["Sector_share_frac"] = final["Sector_share_frac"].clip(lower=0, upper=1)

# =====================================================
# 6) CHECK SHARE SUM BY PRODUCT
# =====================================================
share_check = (
    final.groupby("product", dropna=False)["Sector_share_frac"]
    .sum(min_count=1)
    .reset_index(name="sector_share_sum")
)

share_check["difference_from_1"] = share_check["sector_share_sum"] - 1.0
share_check["needs_attention"] = share_check["difference_from_1"].abs() > tolerance

print("\n===== SHARE CHECK =====")
print("Products with shares not summing to 1:", share_check["needs_attention"].sum())

final = final.merge(share_check, on="product", how="left")

# =====================================================
# 7) APPLY SECTOR SPLIT ONLY WHEN NEEDED
# =====================================================
final["n_sectors_for_product"] = final.groupby("product", dropna=False)["Sector"].transform("nunique")
final["apply_sector_split"] = final["n_sectors_for_product"] > 1

existing_split = [c for c in cols_to_split if c in final.columns]

for c in existing_split:
    final[c] = pd.to_numeric(final[c], errors="coerce").astype(float)
    final[f"{c}_sector"] = final[c].astype(float)

    split_mask = final["apply_sector_split"] & final["Sector_share_frac"].notna()

    final.loc[split_mask, f"{c}_sector"] = (
        final.loc[split_mask, c].astype(float) *
        final.loc[split_mask, "Sector_share_frac"].astype(float)
    )

print("\nApplied sector split to:", existing_split)

# =====================================================
# 8) CHECK PRODUCTS WITH NO EUROSTAT MATCH
# =====================================================
coverage = desc_master.merge(
    eur_wide[["product"]].drop_duplicates(),
    on="product",
    how="left",
    indicator=True,
)

missing_products = coverage.loc[
    coverage["_merge"] == "left_only",
    ["product"]
].drop_duplicates()

if "Full_PRODCOM_name" in desc_master.columns:
    missing_products = missing_products.merge(
        desc_master[["product", "Full_PRODCOM_name"]].drop_duplicates(),
        on="product",
        how="left"
    )

print("Products with no Eurostat match (2005-2022):", len(missing_products))

# =====================================================
# 9) CONSERVATION CHECK
# =====================================================
check_rows = []
for c in existing_split:
    original_total = final[c].sum(skipna=True)
    split_total = final[f"{c}_sector"].sum(skipna=True)

    check_rows.append({
        "variable": c,
        "original_total": original_total,
        "split_total": split_total,
        "difference": split_total - original_total
    })

conservation_check = pd.DataFrame(check_rows)

print("\n===== CONSERVATION CHECK =====")
print(conservation_check)

# =====================================================
# 10) EXAMPLE CHECK
# =====================================================
example_code = "22212153"
example = final.loc[final["product"] == example_code, [
    "product", "Sector", "Sector_share [%]", "Sector_share_frac",
    "n_sectors_for_product", "apply_sector_split"
] + existing_split + [f"{c}_sector" for c in existing_split]]

if not example.empty:
    print(f"\n===== EXAMPLE {example_code} =====")
    print(example)
else:
    print(f"\nExample code {example_code} not found.")

# =====================================================
# 11) LOAD TABLE 24 FROM EXCEL
# =====================================================
table24_df = pd.read_excel(table24_file)
table24_df.columns = [str(c).strip() for c in table24_df.columns]
table24_df = table24_df.rename(columns={table24_df.columns[0]: "Category"})

table24_df.columns = [
    "Category" if c == "Category" else norm_sector(c)
    for c in table24_df.columns
]

table24_used_share = {}

for _, row in table24_df.iterrows():
    category = str(row["Category"]).strip()
    table24_used_share[category] = {}

    for col in table24_df.columns:
        if col == "Category":
            continue

        val = row[col]
        if pd.notna(val) and float(val) > 0:
            table24_used_share[category][col] = normalize_share_value(val)

def get_used_share(semi_category, sector):
    semi_category = str(semi_category).strip()
    sector = norm_sector(sector)

    if semi_category not in table24_used_share:
        return 0.0

    return float(table24_used_share[semi_category].get(sector, 0.0))

# =====================================================
# 12) BUILD MFA DATA FOR TARGET_YEAR
# =====================================================
df = final.copy()
df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce")
df = df[df[YEAR_COL] == TARGET_YEAR].copy()

if df.empty:
    raise ValueError(f"No rows found for {TARGET_YEAR}")

df[SECTOR_COL] = df[SECTOR_COL].map(norm_sector)
df[CODE_COL] = df[CODE_COL].astype(str).str.strip()

for c in [PROD_Q, IMP_Q, EXP_Q, WEIGHT_COL, PLSH_COL, SECSH_COL, CONV_COL]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    else:
        raise KeyError(f"Required column '{c}' not found in data.")

df[PLSH_COL] = df[PLSH_COL].apply(normalize_share_value)
df[SECSH_COL] = df[SECSH_COL].apply(normalize_share_value)

# optional reclassification
carpet_codes = ["13931200", "13931300", "13931990"]
mask_carpets = df[CODE_COL].isin(carpet_codes)
df.loc[mask_carpets, SECTOR_COL] = "Construction"

# =====================================================
# 13) PLASTIC MASS FLOWS
# =====================================================
df["PROD_KG_PRODUCT"] = df[PROD_Q] * df[WEIGHT_COL]
df["IMP_KG_PRODUCT"]  = df[IMP_Q]  * df[WEIGHT_COL]
df["EXP_KG_PRODUCT"]  = df[EXP_Q]  * df[WEIGHT_COL]

df["plastic_frac"] = df[PLSH_COL]
df["sector_frac"]  = df[SECSH_COL]

# Keep your original conversion logic
df["conv_Mt_per_kg"] = df[CONV_COL] * 0.001

# You can include sector_frac here if needed
df["PROD_MT_PL"] = df["PROD_KG_PRODUCT"] * df["conv_Mt_per_kg"] * df["plastic_frac"]
df["IMP_MT_PL"]  = df["IMP_KG_PRODUCT"]  * df["conv_Mt_per_kg"] * df["plastic_frac"]
df["EXP_MT_PL"]  = df["EXP_KG_PRODUCT"]  * df["conv_Mt_per_kg"] * df["plastic_frac"]

# =====================================================
# 14) CLASSIFY
# =====================================================
df["semi_category"] = df[NAME_COL].apply(build_semi_category_from_name)
df["is_semi"] = df[TYPE_COL].apply(is_semi)
df["is_finished"] = df[TYPE_COL].apply(is_finished)

# =====================================================
# 15) APPLY TABLE 24 SPLIT
# =====================================================
mask_semi = df["is_semi"].fillna(False)

df["semi_used_share"] = 0.0
df.loc[mask_semi, "semi_used_share"] = df.loc[mask_semi].apply(
    lambda r: get_used_share(r["semi_category"], r[SECTOR_COL]),
    axis=1
)

df["semi_used_share"] = df["semi_used_share"].clip(0, 1)
df["semi_direct_share"] = 1.0
df.loc[mask_semi, "semi_direct_share"] = 1.0 - df.loc[mask_semi, "semi_used_share"]

for flow in ["PROD_MT_PL", "IMP_MT_PL", "EXP_MT_PL"]:
    df[f"SEMI_TO_FINISHED_{flow}"] = 0.0
    df[f"SEMI_DIRECT_{flow}"] = 0.0

    df.loc[mask_semi, f"SEMI_TO_FINISHED_{flow}"] = (
        df.loc[mask_semi, flow] * df.loc[mask_semi, "semi_used_share"]
    )
    df.loc[mask_semi, f"SEMI_DIRECT_{flow}"] = (
        df.loc[mask_semi, flow] * df.loc[mask_semi, "semi_direct_share"]
    )

# =====================================================
# 16) FINISHED ONLY FLOWS
# =====================================================
df["FINISHED_ONLY_PROD_MT"] = 0.0
df["FINISHED_ONLY_IMP_MT"]  = 0.0
df["FINISHED_ONLY_EXP_MT"]  = 0.0

df.loc[df["is_finished"], "FINISHED_ONLY_PROD_MT"] = df.loc[df["is_finished"], "PROD_MT_PL"]
df.loc[df["is_finished"], "FINISHED_ONLY_IMP_MT"]  = df.loc[df["is_finished"], "IMP_MT_PL"]
df.loc[df["is_finished"], "FINISHED_ONLY_EXP_MT"]  = df.loc[df["is_finished"], "EXP_MT_PL"]

# =====================================================
# 17) BUILD PRODCOM MFA TOTALS
# =====================================================

df["SEMI_DIRECT_CONS_MT"] = 0.0
df.loc[df["is_semi"], "SEMI_DIRECT_CONS_MT"] = (
    df.loc[df["is_semi"], "SEMI_DIRECT_PROD_MT_PL"]
    + df.loc[df["is_semi"], "SEMI_DIRECT_IMP_MT_PL"]
    - df.loc[df["is_semi"], "SEMI_DIRECT_EXP_MT_PL"]
)


df["SEMI_TO_FINISHED_CONS_MT"] = (
    df["SEMI_TO_FINISHED_PROD_MT_PL"]
    + df["SEMI_TO_FINISHED_IMP_MT_PL"]
    - df["SEMI_TO_FINISHED_EXP_MT_PL"]
)

df["FINISHED_TOTAL_PROD_MT"] = (
    df["FINISHED_ONLY_PROD_MT"] + df["SEMI_TO_FINISHED_PROD_MT_PL"]
)


df["FINISHED_TOTAL_EXP_MT"] = (
    df["FINISHED_ONLY_EXP_MT"] + df["SEMI_TO_FINISHED_EXP_MT_PL"]
)

df["FINISHED_TOTAL_IMP_MT"] = (
    df["FINISHED_ONLY_IMP_MT"] + df["SEMI_TO_FINISHED_IMP_MT_PL"]
)

df["FINISHED_EFFECTIVE_CONS_MT"] = (
    df["FINISHED_TOTAL_PROD_MT"]+ df["FINISHED_TOTAL_IMP_MT"] - df["FINISHED_TOTAL_EXP_MT"]
)

df["CONS_FINAL_MT"] = (
    df["FINISHED_EFFECTIVE_CONS_MT"] + df["SEMI_DIRECT_CONS_MT"]
)

# =====================================================
# 18) DIAGNOSTICS
# =====================================================
diag_cols = [
    "SEMI_DIRECT_CONS_MT",
    "SEMI_TO_FINISHED_CONS_MT",
    "FINISHED_EFFECTIVE_CONS_MT",
    "CONS_FINAL_MT"
]

for c in diag_cols:
    df[f"{c}_NEGATIVE"] = df[c] < 0

totals = df[[
    "PROD_MT_PL",
    "IMP_MT_PL",
    "EXP_MT_PL",
    "SEMI_DIRECT_CONS_MT",
    "SEMI_TO_FINISHED_CONS_MT",
    "FINISHED_TOTAL_PROD_MT",
    "FINISHED_TOTAL_IMP_MT",
    "FINISHED_TOTAL_EXP_MT",
    "CONS_FINAL_MT"
]].sum().to_frame("Mt")

# =====================================================
# 19B) PRODUCTION / IMPORT / EXPORT BY PRODUCT TYPE AND SECTOR
# =====================================================

# 1) By sector and type of product
Type_trade_by_sector = (
    df.groupby([SECTOR_COL, TYPE_COL], dropna=False)[
        ["PROD_MT_PL", "IMP_MT_PL", "EXP_MT_PL"]
    ]
    .sum()
    .reset_index()
    .rename(columns={
        "PROD_MT_PL": "Production_MT",
        "IMP_MT_PL": "Import_MT",
        "EXP_MT_PL": "Export_MT"
    })
)

print("\n===== PRODUCTION / IMPORT / EXPORT BY SECTOR AND TYPE =====")
print(Type_trade_by_sector)

# =====================================================
# 19) FINISHED CATEGORY = FINISHED ONLY + SEMI TO FINISHED
# =====================================================

# Finished-only flows by sector
Finished_only_by_sector = (
    df.groupby(SECTOR_COL, dropna=False)[
        ["FINISHED_ONLY_PROD_MT", "FINISHED_ONLY_IMP_MT", "FINISHED_ONLY_EXP_MT"]
    ]
    .sum()
    .reset_index()
    .rename(columns={
        "FINISHED_ONLY_PROD_MT": "Finished_only_production_MT",
        "FINISHED_ONLY_IMP_MT": "Finished_only_import_MT",
        "FINISHED_ONLY_EXP_MT": "Finished_only_export_MT"
    })
)

# Semi-finished flows used in finished production by sector
Semi_to_finished_by_sector = (
    df.groupby(SECTOR_COL, dropna=False)[
        [
            "SEMI_TO_FINISHED_PROD_MT_PL",
            "SEMI_TO_FINISHED_IMP_MT_PL",
            "SEMI_TO_FINISHED_EXP_MT_PL"
        ]
    ]
    .sum()
    .reset_index()
    .rename(columns={
        "SEMI_TO_FINISHED_PROD_MT_PL": "Semi_to_finished_production_MT",
        "SEMI_TO_FINISHED_IMP_MT_PL": "Semi_to_finished_import_MT",
        "SEMI_TO_FINISHED_EXP_MT_PL": "Semi_to_finished_export_MT"
    })
)

# Merge both
Finished_category_by_sector = (
    Finished_only_by_sector
    .merge(Semi_to_finished_by_sector, on=SECTOR_COL, how="outer")
    .fillna(0)
)

# Final finished category = finished only + semi to finished
Finished_category_by_sector["Finished_category_production_MT"] = (
    Finished_category_by_sector["Finished_only_production_MT"]
    + Finished_category_by_sector["Semi_to_finished_production_MT"]
)

Finished_category_by_sector["Finished_category_import_MT"] = (
    Finished_category_by_sector["Finished_only_import_MT"]
    + Finished_category_by_sector["Semi_to_finished_import_MT"]
)

Finished_category_by_sector["Finished_category_export_MT"] = (
    Finished_category_by_sector["Finished_only_export_MT"]
    + Finished_category_by_sector["Semi_to_finished_export_MT"]
)

print("\n===== FINISHED CATEGORY BY SECTOR =====")
print(Finished_category_by_sector)

# =====================================================
# 19D) TOTAL CHECKS
# =====================================================

Finished_category_totals = pd.DataFrame({
    "Flow": ["Production", "Import", "Export"],
    "Finished_only_MT": [
        Finished_category_by_sector["Finished_only_production_MT"].sum(),
        Finished_category_by_sector["Finished_only_import_MT"].sum(),
        Finished_category_by_sector["Finished_only_export_MT"].sum()
    ],
    "Semi_to_finished_MT": [
        Finished_category_by_sector["Semi_to_finished_production_MT"].sum(),
        Finished_category_by_sector["Semi_to_finished_import_MT"].sum(),
        Finished_category_by_sector["Semi_to_finished_export_MT"].sum()
    ],
    "Finished_category_total_MT": [
        Finished_category_by_sector["Finished_category_production_MT"].sum(),
        Finished_category_by_sector["Finished_category_import_MT"].sum(),
        Finished_category_by_sector["Finished_category_export_MT"].sum()
    ]
})

print("\n===== FINISHED CATEGORY TOTALS =====")
print(Finished_category_totals)

# =====================================================
#19 IMPORT / EXPORT OF SEMI AND FINISHED BY SECTOR
# =====================================================
Trade_by_sector_type = (
    df.groupby([SECTOR_COL, TYPE_COL], dropna=False)[["IMP_MT_PL", "EXP_MT_PL"]].sum().reset_index() )
print("\n===== IMPORT / EXPORT BY SECTOR AND TYPE OF PRODUCT =====")
print(Trade_by_sector_type)

# -----------------------------------------------------
# Semi-finished only
# -----------------------------------------------------
Semi_trade_by_sector = (
    df[df["is_semi"]]
    .groupby(SECTOR_COL, dropna=False)[["IMP_MT_PL", "EXP_MT_PL"]]
    .sum()
    .reset_index()
    .rename(columns={
        "IMP_MT_PL": "Import_semi_MT",
        "EXP_MT_PL": "Export_semi_MT"
    })
)
print("\n===== SEMI-FINISHED IMPORT / EXPORT BY SECTOR =====")
print(Semi_trade_by_sector)
# -----------------------------------------------------
# Finished only
# -----------------------------------------------------
Finished_trade_by_sector = (
    df[df["is_finished"]]
    .groupby(SECTOR_COL, dropna=False)[["IMP_MT_PL", "EXP_MT_PL"]]
    .sum()
    .reset_index()
    .rename(columns={
        "IMP_MT_PL": "Import_finished_MT",
        "EXP_MT_PL": "Export_finished_MT"
    })
)
# Merge semi + finished in one table
# -----------------------------------------------------
Trade_summary = (
    pd.merge(Semi_trade_by_sector, Finished_trade_by_sector, on=SECTOR_COL, how="outer")
      .fillna(0)
)

Trade_summary["Import_total_MT"] = (
    Trade_summary["Import_semi_MT"] + Trade_summary["Import_finished_MT"]
)
Trade_summary["Export_total_MT"] = (
    Trade_summary["Export_semi_MT"] + Trade_summary["Export_finished_MT"]
)

print("\n===== FINAL TRADE SUMMARY BY SECTOR =====")
print(Trade_summary)

Trade_totals = pd.DataFrame({
    "Category": ["Semi-finished", "Finished"],
    "Imports_MT": [
        df.loc[df["is_semi"], "IMP_MT_PL"].sum(),
        df.loc[df["is_finished"], "IMP_MT_PL"].sum()
    ],
    "Exports_MT": [
        df.loc[df["is_semi"], "EXP_MT_PL"].sum(),
        df.loc[df["is_finished"], "EXP_MT_PL"].sum()
    ]
})

print("\n===== TOTAL IMPORTS / EXPORTS =====")
print(Trade_totals)

# =====================================================
# 20) MASS BALANCE CHECK
# =====================================================
balance = pd.DataFrame({
    "Metric": [
        "Total production",
        "Total imports",
        "Total exports",
        "Apparent consumption = Prod + Imp - Exp",
        "Computed final consumption"
    ],
    "Mt": [
        df["PROD_MT_PL"].sum(),
        df["IMP_MT_PL"].sum(),
        df["EXP_MT_PL"].sum(),
        df["PROD_MT_PL"].sum() + df["IMP_MT_PL"].sum() - df["EXP_MT_PL"].sum(),
        df["CONS_FINAL_MT"].sum()
    ]
})

balance["Difference_vs_computed"] = np.nan
balance.loc[3, "Difference_vs_computed"] = balance.loc[3, "Mt"] - balance.loc[4, "Mt"]

# =====================================================
# 20) SAVE
# =====================================================
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    final.to_excel(writer, sheet_name="data", index=False)
    missing_products.to_excel(writer, sheet_name="missing_products", index=False)
    share_check.to_excel(writer, sheet_name="share_check", index=False)
    conservation_check.to_excel(writer, sheet_name="conservation_check", index=False)

    df.to_excel(writer, sheet_name=f"mfa_{TARGET_YEAR}", index=False)
    totals.to_excel(writer, sheet_name="mfa_totals")
    balance.to_excel(writer, sheet_name="mass_balance_check", index=False)


    Type_trade_by_sector.to_excel(writer, sheet_name="Type_by_sector", index=False)
    Finished_only_by_sector.to_excel(writer, sheet_name="Finished_only_sector", index=False)
    Semi_to_finished_by_sector.to_excel(writer, sheet_name="Semi_to_finished_sector", index=False)
    Finished_category_by_sector.to_excel(writer, sheet_name="Finished_category_sector", index=False)
    Finished_category_totals.to_excel(writer, sheet_name="Finished_category_totals", index=False)
    
    Trade_by_sector_type.to_excel(writer, sheet_name="Trade_by_sector_type", index=False)
    Semi_trade_by_sector.to_excel(writer, sheet_name="Semi_trade_by_sector", index=False)
    Finished_trade_by_sector.to_excel(writer, sheet_name="Finished_trade_by_sector", index=False)
    Trade_summary.to_excel(writer, sheet_name="Trade_summary_sector", index=False)
    Trade_totals.to_excel(writer, sheet_name="Trade_totals", index=False)

print("Saved:", output_file)
print("\nMFA totals:")
print(totals)
print("\nMass balance check:")
print(balance)

Rows in description file: 577
Unique product codes in description: 458
Filtered Eurostat rows: 393127
Unique Eurostat product codes: 4785
Years: 2005 to 2022
Wide Eurostat rows: 68916
Wide Eurostat columns: ['reporter', 'product', 'TIME_PERIOD', 'EXPQNT', 'EXPVAL', 'IMPQNT', 'IMPVAL', 'PRODQNT', 'PRODVAL', 'QNTUNIT']
Final rows before split: 10011

===== SHARE CHECK =====
Products with shares not summing to 1: 458

Applied sector split to: ['PRODQNT', 'IMPQNT', 'EXPQNT', 'PRODVAL', 'IMPVAL', 'EXPVAL']
Products with no Eurostat match (2005-2022): 1

===== CONSERVATION CHECK =====
  variable  original_total   split_total    difference
0  PRODQNT    1.643962e+13  1.542260e+13 -1.017021e+12
1   IMPQNT    8.920548e+11  7.891256e+11 -1.029292e+11
2   EXPQNT    2.697966e+12  2.550318e+12 -1.476477e+11
3  PRODVAL    1.468471e+13  1.204056e+13 -2.644151e+12
4   IMPVAL    4.808006e+12  4.406660e+12 -4.013460e+11
5   EXPVAL    5.309005e+12  4.698809e+12 -6.101957e+11

===== EXAMPLE 22212153 =====

In [ ]:
import pandas as pd
import numpy as np


# Here, we compute the TC for export and import of finshed and semi-finished from 2005 to 2022 
# ==========================================================
# FILES
# ==========================================================
trade_file = "PRODCOM_2005_2022_EU27_2022.xlsx"
plastic_demand_file = "Plastic_demand_2022.xlsx"
sector_alloc_file   = "Sector_allocation_2022.xlsx"
tc_domestic_file    = "TC_semi_Finished.xlsx"

out_file = "TC_Trade_Finished_Semi_By_Sector_EU_27_2022.xlsx"

# ==========================================================
# SETTINGS
# ==========================================================
trade_sheet = "mfa_2022"
# ==========================================================
# HELPERS
# ==========================================================
def norm_sector(s):
    s = str(s).strip()
    low = s.lower()

    if low in ["clothing_textiles", "clothing & textiles", "textiles", "clothing",
               "textiles and clothing", "clothing and textiles", "textiles&clothing"]:
        return "Clothing and textiles"
    if low in ["building & construction", "buildings and construction", "construction", "building"]:
        return "Construction"
    if low in ["agriculture", "agriculture, farming, gardening"]:
        return "Agriculture"
    if low in ["eee", "electrical and electronic equipment", "electrical/electronics",
               "electrical & electronic equipment"]:
        return "EEE"
    if low in ["healthcare", "health care"]:
        return "Healthcare"
    if low in ["fishing"]:
        return "Fishing"
    if low in ["other", "others"]:
        return "Other"
    if low in ["packaging"]:
        return "Packaging"
    if low in ["transport", "automotive"]:
        return "Transport"

    return s


def product_type_group(x):
    x = str(x).lower().strip()

    if "semi" in x:
        return "semi"
    if "finish" in x and "semi" not in x:
        return "finished"

    return "other"


def to_float(x):
    if pd.isna(x):
        return 0.0

    if isinstance(x, str):
        x = x.strip().replace("%", "")
        if x == "":
            return 0.0

    return float(x)


def clean_text(x):
    if pd.isna(x):
        return ""
    return " ".join(str(x).replace("\n", " ").replace("\r", " ").split()).strip().lower()


sector_to_code = {
    "Packaging": "P",
    "Construction": "C",
    "Transport": "T",
    "EEE": "E",
    "Agriculture": "A",
    "Clothing and textiles": "C&T",
    "Healthcare": "H",
    "Fishing": "F",
    "Other": "O"
}

sector_order = ["P", "C", "T", "E", "A", "C&T", "H", "F", "O"]

code_to_sector = {v: k for k, v in sector_to_code.items()}

# ==========================================================
# 1) LOAD TRADE DATA
# ==========================================================
trade = pd.read_excel(trade_file, sheet_name=trade_sheet)
trade.columns = trade.columns.str.strip()

required_trade_cols = ["Sector", "Type of product", "IMP_MT_PL", "EXP_MT_PL"]
missing_trade = [c for c in required_trade_cols if c not in trade.columns]

if missing_trade:
    raise KeyError(f"Missing columns in trade file: {missing_trade}")

trade = trade[required_trade_cols].copy()

trade["Sector"] = trade["Sector"].apply(norm_sector)
trade["Sector_code"] = trade["Sector"].map(sector_to_code)

trade["Type_group"] = trade["Type of product"].apply(product_type_group)

trade["IMP_MT_PL"] = pd.to_numeric(trade["IMP_MT_PL"], errors="coerce").fillna(0.0)
trade["EXP_MT_PL"] = pd.to_numeric(trade["EXP_MT_PL"], errors="coerce").fillna(0.0)

# Keep only semi and finished
trade = trade[trade["Type_group"].isin(["semi", "finished"])].copy()

# Aggregate trade by sector + type
trade_agg = (
    trade
    .groupby(["Sector", "Sector_code", "Type_group"], dropna=False)[["IMP_MT_PL", "EXP_MT_PL"]]
    .sum()
    .reset_index()
)

# ==========================================================
# 2) BUILD DOMESTIC INPUT DATA FROM:
# Plastic demand × sector allocation share × TC
# ==========================================================

# ----------------------------------------------------------
# 2.1 Load plastic demand
# ----------------------------------------------------------
plastic_demand_df = pd.read_excel(plastic_demand_file)
plastic_demand_df.columns = plastic_demand_df.columns.str.strip()

# Try to detect the demand column automatically
possible_demand_cols = [
    "Plastic_demand_MT",
    "Plastic demand",
    "Plastic demand MT",

]

demand_col = None
for c in possible_demand_cols:
    if c in plastic_demand_df.columns:
        demand_col = c
        break

if demand_col is None:
    numeric_cols = plastic_demand_df.select_dtypes(include="number").columns.tolist()
    if len(numeric_cols) == 0:
        raise KeyError("No plastic demand column found in Plastic_demand.xlsx.")
    demand_col = numeric_cols[0]

Plastic_demand_MT = float(plastic_demand_df[demand_col].dropna().iloc[0])

print("\nPlastic demand used:", Plastic_demand_MT, "Mt")

# ----------------------------------------------------------
# 2.2 Load allocation shares by sector
alloc = pd.read_excel(sector_alloc_file, header=None)

# sector names are in row 0, from column B onward
sector_names = alloc.iloc[0, 1:].dropna().tolist()

# allocation values are in row 1, from column B onward
allocation_values = alloc.iloc[1, 1:1 + len(sector_names)].tolist()

sector_alloc = pd.DataFrame({
    "Sector": sector_names,
    "Allocation_share": allocation_values
})

sector_alloc["Sector"] = sector_alloc["Sector"].apply(norm_sector)
sector_alloc["Sector_code"] = sector_alloc["Sector"].map(sector_to_code)

sector_alloc["Allocation_share"] = pd.to_numeric(
    sector_alloc["Allocation_share"], errors="coerce"
).fillna(0.0)

# If shares are written as percentages, convert to fractions
sector_alloc["Allocation_share"] = np.where(
    sector_alloc["Allocation_share"] > 1,
    sector_alloc["Allocation_share"] / 100,
    sector_alloc["Allocation_share"]
)

sector_alloc = sector_alloc[sector_alloc["Sector_code"].isin(sector_order)].copy()

print("\n===== SECTOR ALLOCATION SHARES =====")
print(sector_alloc)

# ----------------------------------------------------------
# 2.3 Load TC file for domestic split: plastic -> semi / finished
# ----------------------------------------------------------
tc_dom = pd.read_excel(tc_domestic_file)
tc_dom.columns = tc_dom.columns.str.strip()

required_tc_cols = ["FROM", "TO"]
missing_tc_cols = [c for c in required_tc_cols if c not in tc_dom.columns]

if missing_tc_cols:
    raise KeyError(f"Missing columns in TC file: {missing_tc_cols}")

tc_dom["_FROM_CLEAN"] = tc_dom["FROM"].apply(clean_text)
tc_dom["_TO_CLEAN"] = tc_dom["TO"].apply(clean_text)

semi_mask = (
    (tc_dom["_FROM_CLEAN"] == "plastic products manufacturing") &
    (tc_dom["_TO_CLEAN"] == "semi-finished products manufacturing")
)

finished_mask = (
    (tc_dom["_FROM_CLEAN"] == "plastic products manufacturing") &
    (tc_dom["_TO_CLEAN"] == "finished products manufacturing")
)

if not semi_mask.any():
    raise ValueError("Row not found: Plastic products manufacturing -> Semi-finished products manufacturing")

if not finished_mask.any():
    raise ValueError("Row not found: Plastic products manufacturing -> Finished products manufacturing")

semi_row = tc_dom.loc[semi_mask].iloc[0]
finished_row = tc_dom.loc[finished_mask].iloc[0]

# Build a flexible sector-column map from TC file columns
tc_sector_col_map = {}

for col in tc_dom.columns:
    if col in ["FROM", "TO", "_FROM_CLEAN", "_TO_CLEAN"]:
        continue

    col_clean = norm_sector(col)

    if col_clean in sector_to_code:
        tc_sector_col_map[sector_to_code[col_clean]] = col
    elif col in sector_order:
        tc_sector_col_map[col] = col

print("\n===== TC SECTOR COLUMN MAP =====")
print(tc_sector_col_map)

missing_tc_sector_cols = [s for s in sector_order if s not in tc_sector_col_map]
if missing_tc_sector_cols:
    raise KeyError(
        f"These sector columns are missing in TC file: {missing_tc_sector_cols}. "
        f"Available TC columns are: {tc_dom.columns.tolist()}"
    )

# ----------------------------------------------------------
# 2.4 Build domestic input table
# ----------------------------------------------------------
rows = []

for _, r in sector_alloc.iterrows():

    sector_code = r["Sector_code"]
    sector_name = code_to_sector.get(sector_code, r.get("Sector", sector_code))
    alloc_share = float(r["Allocation_share"])

    tc_col = tc_sector_col_map[sector_code]

    tc_semi = to_float(semi_row[tc_col])
    tc_finished = to_float(finished_row[tc_col])

    # Convert TC if written as %
    if tc_semi > 1:
        tc_semi = tc_semi / 100

    if tc_finished > 1:
        tc_finished = tc_finished / 100

    semi_dom = Plastic_demand_MT * alloc_share * tc_semi
    finished_dom = Plastic_demand_MT * alloc_share * tc_finished

    rows.append({
        "Sector": sector_name,
        "Sector_code": sector_code,
        "Plastic_demand_MT": Plastic_demand_MT,
        "Allocation_share": alloc_share,
        "TC_plastic_to_semi": tc_semi,
        "TC_plastic_to_finished": tc_finished,
        "Semi_dom_MT": semi_dom,
        "Finished_domestic_MT": finished_dom
    })

dom_use_detail = pd.DataFrame(rows)

dom_use = (
    dom_use_detail
    .groupby("Sector_code", dropna=False)[["Semi_dom_MT", "Finished_domestic_MT"]]
    .sum()
    .reset_index()
)

print("\n===== DOMESTIC INPUTS BUILT FROM DEMAND × ALLOCATION × TC =====")
print(dom_use_detail)

# ==========================================================
# 3) MERGE TRADE + DOMESTIC INPUT
# ==========================================================
tc = trade_agg.merge(dom_use, on="Sector_code", how="left")

tc["Domestic_input_MT"] = np.where(
    tc["Type_group"] == "semi",
    tc["Semi_dom_MT"],
    tc["Finished_domestic_MT"]
)
# ==========================================================
# 4) COMPUTE JRC TRADE TCs
# ==========================================================

tc["TC_Import"] = np.where(
    tc["Domestic_input_MT"] > 0,
    tc["IMP_MT_PL"] / tc["Domestic_input_MT"],
    np.nan
)

tc["TC_Export"] = np.where(
    tc["Domestic_input_MT"] > 0,
    tc["EXP_MT_PL"] / tc["Domestic_input_MT"],
    np.nan
)

tc["TC_Import"] = tc["TC_Import"].round(2)
tc["TC_Export"] = tc["TC_Export"].round(2)

# Add readable type
tc["Type of product"] = np.where(
    tc["Type_group"] == "semi",
    "Semi-finished",
    "Finished"
)

# ==========================================================
# 5) LONG FORMAT TABLE
# ==========================================================
tc_final = tc[[
    "Sector",
    "Sector_code",
    "Type of product",
    "Type_group",
    "IMP_MT_PL",
    "EXP_MT_PL",
    "Domestic_input_MT",
    "TC_Import",
    "TC_Export"
]].sort_values(["Sector_code", "Type_group"]).reset_index(drop=True)

print("\n===== LONG TABLE: JRC TRADE TCs BY SECTOR =====")
print(tc_final)

# ==========================================================
# 6) BUILD JRC-STYLE MATRIX
# ==========================================================
jrc_rows = [
    (
        "Import",
        "Semi-finished products manufacturing",
        "TC_Import",
        "semi"
    ),
    (
        "Import",
        "Finished products manufacturing",
        "TC_Import",
        "finished"
    ),
    (
        "Semi-finished products manufacturing",
        "Export",
        "TC_Export",
        "semi"
    ),
    (
        "Finished products manufacturing",
        "Export",
        "TC_Export",
        "finished"
    ),
    (
        "Plastic products manufacturing",
        "Semi-finished products manufacturing (domestic input Semi_dom_MT)",
        "Domestic_input_MT",
        "semi"
    ),
    (
        "Plastic products manufacturing",
        "Finished products manufacturing (domestic input Finished_domestic_MT)",
        "Domestic_input_MT",
        "finished"
    )
]

matrix_rows = []

for from_node, to_node, value_col, group in jrc_rows:

    row = {
        "FROM": from_node,
        "TO": to_node
    }

    sub = tc_final[tc_final["Type_group"] == group].copy()

    for sec in sector_order:
        value = sub.loc[sub["Sector_code"] == sec, value_col]

        if len(value) > 0:
            row[sec] = value.iloc[0]
        else:
            row[sec] = np.nan

    matrix_rows.append(row)

jrc_matrix = pd.DataFrame(matrix_rows)

print("\n===== JRC-STYLE MATRIX =====")
print(jrc_matrix)

# ==========================================================
# 7) OPTIONAL: SEPARATE TABLES
# ==========================================================
semi_table = tc_final[tc_final["Type_group"] == "semi"].copy()
finished_table = tc_final[tc_final["Type_group"] == "finished"].copy()

# ==========================================================
# 8) SAVE OUTPUT
# ==========================================================
with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    tc_final.to_excel(writer, sheet_name="TC_trade_long", index=False)
    semi_table.to_excel(writer, sheet_name="TC_semi", index=False)
    finished_table.to_excel(writer, sheet_name="TC_finished", index=False)
    jrc_matrix.to_excel(writer, sheet_name="JRC_style_matrix", index=False)
    dom_use_detail.to_excel(writer, sheet_name="Domestic_inputs_calc", index=False)

print(f"\nSaved: {out_file}")